# Rodzina domowa

**Pytanie:** która marka ETF ma udawać „nasze” fundusze, tak żeby zadanie „kto dołoży pieniądze w następnym kwartale” było dla modelu sensowne?

**Wniosek:** JPMorgan (prefiksy CUSIP `46641Q`, `46654Q`). Pozytywów jest kilka procent, a nie kilkadziesiąt jak przy iShares czy Vanguard. JPMorgan to też aktywny zarządzający z linią ETF, podobnie jak UBS Asset Management. Decyzja i próg kwoty: `docs/decisions.md`.

Pozytyw: filer, który w kwartale t+1 dołożył do marki więcej niż próg nowych pieniędzy, liczonych jako zmiana liczby akcji (po splitach) × cena z t+1. Sam wzrost kursu się nie liczy.

In [1]:
import tempfile
from pathlib import Path

import duckdb

from radar.splits import splits

con = duckdb.connect()
con.sql("SET preserve_insertion_order = false")
%load_ext sql
%sql con
%config SqlMagic.displaylimit = 30

POSITIONS = "../data/processed/positions_pit.parquet"
INFOTABLE = "../data/processed/infotable/*.parquet"
SUBMISSION = "../data/processed/submission/*.parquet"

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

## 1. Marki i ich prefiksy

Nazwa emitenta to wolny tekst, więc służy tylko do znalezienia prefiksów CUSIP (pierwsze 6 znaków to emitent, czyli trust). Prefiks należy do marki, jeśli co najmniej połowa jego wierszy ma nazwę marki. Jeden kwartał wystarczy, bo prefiksy są stałe.

Akcje samych firm (np. JPMorgan Chase, Charles Schwab Corp) usuwamy ręcznie: pasują do nazwy, ale to nie ETF-y.

In [2]:
BRANDS = {
    "iShares": "ISHARES", "Vanguard": "VANGUARD", "SPDR": "SPDR|SELECT SECTOR",
    "Invesco": "INVESCO|POWERSHARES", "Schwab": "SCHWAB", "First Trust": "FIRST TR",
    "Dimensional": "DIMENSIONAL", "JPMorgan": "J ?P ?MORGAN", "VanEck": "VANECK",
    "WisdomTree": "WISDOMTREE", "Global X": "GLOBAL X", "ProShares": "PROSHARES",
    "Fidelity": "FIDELITY", "Avantis": "AMERICAN CENTY|AVANTIS", "PIMCO": "PIMCO",
    "Goldman": "GOLDMAN SACHS ETF", "Capital Group": "CAPITAL GR",
}
NOT_ETF = ["46625H", "48133Q", "808513", "808515", "31620M", "31620R", "31609R", "030371", "G491BT"]

con.sql("CREATE OR REPLACE TABLE brand_re (brand VARCHAR, re VARCHAR)")
con.executemany("INSERT INTO brand_re VALUES (?, ?)", list(BRANDS.items()))
con.sql(f"""
CREATE OR REPLACE TABLE names AS
SELECT upper(i.NAMEOFISSUER) AS name, left(upper(i.CUSIP), 6) AS prefix
FROM '{INFOTABLE}' i JOIN '{SUBMISSION}' s USING (ACCESSION_NUMBER)
WHERE s.PERIODOFREPORT = '2026-03-31' AND s.SUBMISSIONTYPE = '13F-HR'
  AND i.PUTCALL IS NULL AND i.SSHPRNAMTTYPE = 'SH'
""")
con.sql(f"""
CREATE OR REPLACE TABLE families AS
WITH p AS (SELECT prefix, count(*) AS n FROM names GROUP BY 1),
m AS (
    SELECT b.brand, n.prefix, count(*) AS k
    FROM names n JOIN brand_re b ON regexp_matches(n.name, b.re)
    GROUP BY 1, 2
)
SELECT m.brand, m.prefix FROM m JOIN p USING (prefix)
WHERE m.k >= 0.5 * p.n AND m.k >= 50 AND m.prefix NOT IN {tuple(NOT_ETF)}
""")
con.sql("SELECT brand, count(*) AS prefixes FROM families GROUP BY 1 ORDER BY 2 DESC")

brandVARCHAR,prefixesBIGINT
First Trust,44
Invesco,28
iShares,19
Vanguard,19
PIMCO,15
Capital Group,14
ProShares,9
SPDR,9
Fidelity,7
VanEck,5


## 2. Splity papierów marek

Splity liczymy funkcją z projektu, tylko na papierach marek. Wynik jest taki sam jak dla całego pliku, bo split zależy wyłącznie od posiadaczy danego papieru.

In [3]:
tmp = Path(tempfile.mkdtemp())
con.sql(f"""
COPY (
    SELECT p.* FROM '{POSITIONS}' p
    WHERE left(p.CUSIP, 6) IN (SELECT prefix FROM families)
) TO '{tmp / "family_positions.parquet"}' (FORMAT parquet)
""")
split_rel = splits(con, tmp / "family_positions.parquet")
con.sql("CREATE OR REPLACE TABLE family_splits AS SELECT CUSIP, PERIODOFREPORT, split_factor FROM split_rel")
con.sql("SELECT count(*) AS splity FROM family_splits")

splityBIGINT
94


## 3. Przepływy: nowe pieniądze na filera, markę i kwartał

Porównujemy kwartał t z t+1 tylko dla filerów obecnych w obu. Papier, którego filer nie ma w jednym z kwartałów, ma tam 0 akcji. Liczba akcji z t jest mnożona przez faktor splitu z t+1. Okres: przejścia od 2024-12-31 do 2026-03-31.

In [4]:
%%sql
CREATE OR REPLACE TABLE flows AS
WITH filers AS (
    SELECT DISTINCT CIK, PERIODOFREPORT AS t
    FROM '../data/processed/positions_pit.parquet'
    WHERE PERIODOFREPORT >= '2024-12-31'
),
pairs AS (
    SELECT a.CIK, a.t, b.t AS t1
    FROM filers a JOIN filers b ON a.CIK = b.CIK AND b.t = last_day(a.t + INTERVAL 3 MONTH)
),
pos AS (
    SELECT p.CIK, p.PERIODOFREPORT AS t, p.CUSIP, p.shares, p.value_corrected AS v, f.brand
    FROM '../data/processed/positions_pit.parquet' p
    JOIN families f ON left(p.CUSIP, 6) = f.prefix
    WHERE p.PERIODOFREPORT >= '2024-12-31'
),
price AS (
    SELECT CUSIP, t, median(v * 1.0 / shares) AS px FROM pos WHERE shares > 0 AND v > 0 GROUP BY 1, 2
),
a AS (SELECT p.*, pr.t1 FROM pos p JOIN pairs pr USING (CIK, t)),
b AS (
    SELECT p.CIK, pr.t, p.t AS t1, p.CUSIP, p.shares, p.brand
    FROM pos p JOIN pairs pr ON p.CIK = pr.CIK AND p.t = pr.t1
),
j AS (
    SELECT
        coalesce(a.CIK, b.CIK) AS CIK, coalesce(a.t, b.t) AS t, coalesce(a.t1, b.t1) AS t1,
        coalesce(a.brand, b.brand) AS brand, coalesce(a.CUSIP, b.CUSIP) AS CUSIP,
        coalesce(a.shares, 0) AS s0, coalesce(b.shares, 0) AS s1, coalesce(a.v, 0) AS v0
    FROM a FULL JOIN b ON a.CIK = b.CIK AND a.t = b.t AND a.CUSIP = b.CUSIP
)
SELECT j.CIK, j.t, j.brand, sum(j.v0) AS held_v0,
       sum((j.s1 - j.s0 * coalesce(s.split_factor, 1)) * px.px) AS new_money,
       (SELECT count(*) FROM pairs WHERE pairs.t = j.t) AS filers
FROM j
JOIN price px ON px.CUSIP = j.CUSIP AND px.t = j.t1
LEFT JOIN family_splits s ON s.CUSIP = j.CUSIP AND s.PERIODOFREPORT = j.t1
GROUP BY j.CIK, j.t, j.brand

Running query in 'DuckDBPyConnection'

280353 rows affected.

Count
280353


## 4. Porównanie marek (próg 1 mln USD)

Kryterium: odsetek pozytywów wśród wszystkich filerów kilka procent. Przy kilkudziesięciu procentach dokłada prawie każdy duży filer i model uczy się tylko wielkości.

In [5]:
%%sql
WITH q AS (
    SELECT brand, t, any_value(filers) AS filers,
           count(*) FILTER (WHERE held_v0 > 0) AS holders,
           count(*) FILTER (WHERE new_money > 1e6) AS pos,
           count(*) FILTER (WHERE new_money > 1e6 AND held_v0 > 0) AS pos_holders,
           count(*) FILTER (WHERE new_money > 1e6 AND held_v0 <= 0) AS pos_prospects
    FROM flows GROUP BY brand, t
)
SELECT brand,
       round(avg(holders)) AS posiadacze,
       round(avg(pos)) AS pozytywy,
       round(100 * avg(pos / filers), 1) AS proc_wszystkich,
       round(100 * avg(pos_holders / holders), 1) AS proc_posiadaczy,
       round(100 * avg(pos_prospects / (filers - holders)), 2) AS proc_prospektow
FROM q GROUP BY brand ORDER BY posiadacze DESC

Running query in 'DuckDBPyConnection'

brand,posiadacze,pozytywy,proc_wszystkich,proc_posiadaczy,proc_prospektow
iShares,5522.0,2118.0,27.1,37.0,3.27
SPDR,5213.0,1407.0,18.0,25.6,2.82
Vanguard,4939.0,1891.0,24.1,37.2,1.74
Invesco,4365.0,1022.0,13.0,21.8,2.03
Schwab,3172.0,640.0,8.2,19.0,0.76
VanEck,2383.0,399.0,5.1,13.9,1.2
JPMorgan,2363.0,676.0,8.6,26.8,0.82
First Trust,2339.0,490.0,6.2,19.5,0.59
WisdomTree,2103.0,298.0,3.8,12.5,0.61
Dimensional,1929.0,632.0,8.1,31.0,0.58


## 5. JPMorgan: jakie papiery i jaki próg kwoty?

Prefiksy `46641Q` i `46654Q` to wyłącznie „J P MORGAN EXCHANGE TRADED F”. Inne papiery z „JPMorgan” w nazwie to akcje i obligacje banku, fundusze inwestycyjne albo ETF-y innych marek na indeksach JPM.

In [6]:
%%sql
SELECT left(upper(CUSIP), 6) AS prefix, count(DISTINCT upper(CUSIP)) AS papiery, mode(upper(NAMEOFISSUER)) AS nazwa
FROM '../data/processed/infotable/*.parquet'
WHERE left(upper(CUSIP), 6) IN ('46641Q', '46654Q')
GROUP BY 1

Running query in 'DuckDBPyConnection'

prefix,papiery,nazwa
46654Q,49,J P MORGAN EXCHANGE TRADED F
46641Q,68,J P MORGAN EXCHANGE TRADED F


In [7]:
%%sql
WITH thresholds(x) AS (VALUES (1e5), (5e5), (1e6), (5e6), (1e7)),
q AS (
    SELECT x, t, any_value(filers) AS filers,
           count(*) FILTER (WHERE held_v0 > 0) AS holders,
           count(*) FILTER (WHERE new_money > x) AS pos,
           count(*) FILTER (WHERE new_money > x AND held_v0 > 0) AS pos_holders,
           count(*) FILTER (WHERE new_money > x AND held_v0 <= 0) AS pos_prospects
    FROM flows, thresholds WHERE brand = 'JPMorgan' GROUP BY x, t
)
SELECT x AS prog_usd,
       round(avg(pos)) AS pozytywy_na_kwartal,
       round(100 * avg(pos / filers), 1) AS proc_wszystkich,
       round(100 * avg(pos_holders / holders), 1) AS proc_posiadaczy,
       round(100 * avg(pos_prospects / (filers - holders)), 2) AS proc_prospektow
FROM q GROUP BY x ORDER BY x

Running query in 'DuckDBPyConnection'

prog_usd,pozytywy_na_kwartal,proc_wszystkich,proc_posiadaczy,proc_prospektow
100000.0,1249.0,16.0,47.9,2.19
500000.0,872.0,11.2,34.3,1.17
1000000.0,676.0,8.6,26.8,0.82
5000000.0,277.0,3.5,11.0,0.33
10000000.0,170.0,2.2,6.8,0.19


## 6. Decyzja

Próg kwoty X wybierasz Ty: wpisz go do `config/house.yaml` razem z prefiksami i uzasadnij w `docs/decisions.md`. Ustal go na okresach treningowych, nie testowych.